# 03 — Baseline Models: Results

Re-runs the four baselines from `src/models/baselines.py` directly (function calls, not subprocess) since each finishes in minutes -- the cell output itself is the durable record once this notebook is saved. Each isolates a different input modality, so together they show what title-only / image-only / naive-combination signal looks like *before* the main SigLIP2 model (see `02_stage_c_training.ipynb`).

**`text_only_bert` takes ~1.5-2h (full AlephBERT fine-tune) and uses the same MPS GPU as Stage C training. Do not run that section while `02_stage_c_training.ipynb` is training -- run it before starting Stage C, or after it finishes.**

In [ ]:
import sys
sys.path.append("..")

import pandas as pd

from src.utils import load_config, set_seed
from src.models.baselines import tfidf_logreg, text_only_bert, image_only, title_image_frozen

cfg = load_config("../configs/base_config.yaml")
set_seed(cfg["seed"])
k_values = tuple(cfg["evaluation"]["precision_at_k_values"])

train_df = pd.read_csv(f"../{cfg['data']['processed_dir']}/train.csv")
val_df = pd.read_csv(f"../{cfg['data']['processed_dir']}/val.csv")
images_dir = f"../{cfg['data']['images_dir']}"

all_results = {}


## 1. `tfidf_logreg` -- classical-ML floor (title text only, no deep learning)

In [ ]:
metrics, _ = tfidf_logreg(train_df, val_df, k_values=k_values)
all_results["tfidf_logreg"] = metrics
pd.Series(metrics)


## 2. `image_only` -- frozen SigLIP2 vision embeddings + LogisticRegression (no text)

In [ ]:
metrics, _ = image_only(train_df, val_df, images_dir=images_dir, k_values=k_values)
all_results["image_only"] = metrics
pd.Series(metrics)


## 3. `title_image_frozen` -- frozen SigLIP2 text+image embeddings, concatenated, one LogisticRegression

The naive-combination check: does just concatenating both modalities' frozen embeddings beat either alone? (Spoiler from the original run: no -- it lands roughly at `image_only`'s level, below `text_only_bert`. That result is exactly why the main model needs real joint fine-tuning, not just concatenation.)

In [ ]:
metrics, _ = title_image_frozen(train_df, val_df, images_dir=images_dir, k_values=k_values)
all_results["title_image_frozen"] = metrics
pd.Series(metrics)


## 4. `text_only_bert` -- AlephBERT fine-tuned on title+tags

**~1.5-2 hours. Uses the GPU. Do not run concurrently with Stage C training (`02_stage_c_training.ipynb`) -- run this section on its own.**

First attempt (2 epochs, default LR) undertrained badly (F1=0, ROC-AUC below `tfidf_logreg`). The settings below (3 epochs, explicit `learning_rate=3e-5`, `warmup_ratio=0.1`) are what actually worked.

In [ ]:
metrics, _ = text_only_bert(train_df, val_df, k_values=k_values,
                            num_epochs=3, learning_rate=3e-5, warmup_ratio=0.1)
all_results["text_only_bert"] = metrics
pd.Series(metrics)


## 5. All results, saved for reuse

In [ ]:
import os

results_df = pd.DataFrame(all_results).T
os.makedirs("../experiments", exist_ok=True)
results_df.to_csv("../experiments/baselines_results.csv")
results_df.sort_values("pr_auc", ascending=False)
